## 01-Classical Classifier Baseline
In this notebook, we will do a simple baseline classification using classicial method (Tfidf and logistic regression)

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


In [2]:
train_fname = "../data/splits/train.csv"
dev_fname = "../data/splits/dev.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 132
Dev df = 34


In [3]:
train_df.head()

,text,target
0,Reaching for a cookie chair possibly moving to...,CN
1,As a picture little girl and a little boy stan...,AD
2,Tell me everything that you see happening. Eve...,CN
3,I think what I want you to do is take a look a...,CN
4,Tell me all the action that you can say on wha...,AD


In [4]:
dev_df.head()

,text,target
0,"Everything you see going on in this picture, g...",CN
1,"Now you just look at the picture. Okay, this i...",CN
2,Tell me everything that you see going on in th...,AD
3,Girls and boys getting her one that stores sto...,AD
4,"All right, we have a few more tasks to go thro...",CN


### Pre-processing
We will lowercase the text for better efficiency

In [5]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

In [6]:
train_df.head()

,text,target,text_lower
0,Reaching for a cookie chair possibly moving to...,CN,reaching for a cookie chair possibly moving to...
1,As a picture little girl and a little boy stan...,AD,as a picture little girl and a little boy stan...
2,Tell me everything that you see happening. Eve...,CN,tell me everything that you see happening. eve...
3,I think what I want you to do is take a look a...,CN,i think what i want you to do is take a look a...
4,Tell me all the action that you can say on wha...,AD,tell me all the action that you can say on wha...


In [7]:
dev_df.head()

,text,target,text_lower
0,"Everything you see going on in this picture, g...",CN,"everything you see going on in this picture, g..."
1,"Now you just look at the picture. Okay, this i...",CN,"now you just look at the picture. okay, this i..."
2,Tell me everything that you see going on in th...,AD,tell me everything that you see going on in th...
3,Girls and boys getting her one that stores sto...,AD,girls and boys getting her one that stores sto...
4,"All right, we have a few more tasks to go thro...",CN,"all right, we have a few more tasks to go thro..."


### Build Tfidf Vectorizer

We can use Tfidf Vectorizer to convert text to a fixed-length vector

In [8]:
vectorizer = TfidfVectorizer(max_features=2000)

corpus = train_df["text_lower"]
print("Building Tfidf Vectorizer...")
X_train = vectorizer.fit_transform(corpus)
print("Completed")

Building Tfidf Vectorizer...
Completed


In [9]:
print(X_train.shape)

(132, 1221)


### Train Logistic regression using Tfidf vectorizer

We can now train a simple logistic regression classifier using processed X_train

In [10]:
y_train = train_df["target"]

print("Training logistic classifier...")
clf = LogisticRegression(random_state=42, max_iter=200).fit(X_train, y_train)
print("Completed")

Training logistic classifier...
Completed


In [11]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 93.94 %


### Get accuracy using cross validation

In [12]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=5)
print("Completed")
print(scores)
print("Average =", np.average(scores))

Performing cross-validation...
Completed
[0.77777778 0.85185185 0.84615385 0.73076923 0.61538462]
Average = 0.7643874643874644


In [13]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.95      0.92      0.94        66
          CN       0.93      0.95      0.94        66

    accuracy                           0.94       132
   macro avg       0.94      0.94      0.94       132
weighted avg       0.94      0.94      0.94       132



### Now we can evaluate on dev set

In [14]:
X_dev = vectorizer.transform(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (34, 1221)
y_dev len = 34


In [15]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 73.53 %


In [16]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.80      0.76      0.78        21
          CN       0.64      0.69      0.67        13

    accuracy                           0.74        34
   macro avg       0.72      0.73      0.72        34
weighted avg       0.74      0.74      0.74        34



The baseline result is reasonable for dev set (74% accuracy, 78% recall). 
However, it's much lower than the scores for train set (94% for accuracy, 92% for recall). It means our model is likely to overfit on train data. 